<a href="https://colab.research.google.com/github/amitpaudel21/ARTIFICIAL_INTELLIGENCE_LAB/blob/main/1BF24CS042_WEEK7_FORWARD_CHAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from itertools import count

# ---------------------------------------------------------------- helpers
def is_var(t):
    return isinstance(t, str) and t.startswith("?")


def substitute(atom, theta):
    return tuple(theta.get(t, t) if is_var(t) else t for t in atom)


def unify_atom(pattern, fact, theta):
    """Match a rule premise (may hold variables) against a ground fact."""
    if pattern[0] != fact[0] or len(pattern) != len(fact):
        return None
    theta = dict(theta)
    for p, f in zip(pattern[1:], fact[1:]):
        p = theta.get(p, p) if is_var(p) else p
        if is_var(p):
            theta[p] = f
        elif p != f:
            return None
    return theta


def match_premises(premises, facts, theta):
    """Yield every substitution that satisfies all premises (conjunction)."""
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for fact in facts:
        t2 = unify_atom(first, fact, theta)
        if t2 is not None:
            yield from match_premises(rest, facts, t2)


def fmt(atom):
    return f"{atom[0]}({', '.join(atom[1:])})"


# ---------------------------------------------------------- knowledge base
# 'Owns(A, x) ^ Missile(x)'  with  EXISTENTIAL INSTANTIATION  x -> T1
facts = [
    ("Owns", "A", "T1"),          # from  Ex Owns(A,x) ^ Missile(x)
    ("Missile", "T1"),
    ("American", "Robert"),
    ("Enemy", "A", "America"),
]

rules = [
    # R1: American(p) ^ Weapon(q) ^ Sells(p,q,r) ^ Hostile(r) => Criminal(p)
    ([("American", "?p"), ("Weapon", "?q"), ("Sells", "?p", "?q", "?r"),
      ("Hostile", "?r")], ("Criminal", "?p")),
    # R2: Missile(x) ^ Owns(A,x) => Sells(Robert,x,A)
    ([("Missile", "?x"), ("Owns", "A", "?x")], ("Sells", "Robert", "?x", "A")),
    # R3: Missile(x) => Weapon(x)
    ([("Missile", "?x")], ("Weapon", "?x")),
    # R4: Enemy(x, America) => Hostile(x)
    ([("Enemy", "?x", "America")], ("Hostile", "?x")),
]
rule_names = ["R1 (crime law)", "R2 (missiles sold)", "R3 (missile=>weapon)",
              "R4 (enemy=>hostile)"]


# ------------------------------------------------------- forward chaining
def forward_chain(facts, rules, query):
    """FOL-FC-ASK: returns (proved?, derivation log)."""
    kb = list(facts)
    # provenance: fact -> (rule name, [supporting facts])
    why = {f: ("Given", []) for f in facts}
    log = []
    for it in count(1):
        new = []
        for (premises, concl), name in zip(rules, rule_names):
            for theta in match_premises(premises, kb, {}):
                inferred = substitute(concl, theta)
                if inferred not in kb and inferred not in new:
                    support = [substitute(p, theta) for p in premises]
                    new.append(inferred)
                    why[inferred] = (name, support)
                    log.append((it, name, support, inferred))
        if not new:                       # fixed point, nothing more derivable
            return query in kb, log, why
        kb.extend(new)
        if query in kb:
            return True, log, why


def print_tree(fact, why, prefix="", last=True, root=True):
    rule, support = why[fact]
    branch = "" if root else ("└── " if last else "├── ")
    tag = "" if rule == "Given" else f"   <= {rule}"
    print(prefix + branch + fmt(fact) + tag)
    ext = prefix + ("" if root else ("    " if last else "│   "))
    for i, s in enumerate(support):
        print_tree(s, why, ext, i == len(support) - 1, root=False)


if __name__ == "__main__":
    query = ("Criminal", "Robert")
    print("Initial facts:")
    for f in facts:
        print("  ", fmt(f))
    print("\nForward chaining iterations:")
    ok, log, why = forward_chain(facts, rules, query)
    for it, name, support, inf in log:
        print(f"  Iter {it}: {name}: "
              f"{' ^ '.join(fmt(s) for s in support)}  =>  {fmt(inf)}")
    print(f"\nQuery {fmt(query)} proved: {ok}\n")
    print("Proof tree:")
    print_tree(query, why)

Initial facts:
   Owns(A, T1)
   Missile(T1)
   American(Robert)
   Enemy(A, America)

Forward chaining iterations:
  Iter 1: R2 (missiles sold): Missile(T1) ^ Owns(A, T1)  =>  Sells(Robert, T1, A)
  Iter 1: R3 (missile=>weapon): Missile(T1)  =>  Weapon(T1)
  Iter 1: R4 (enemy=>hostile): Enemy(A, America)  =>  Hostile(A)
  Iter 2: R1 (crime law): American(Robert) ^ Weapon(T1) ^ Sells(Robert, T1, A) ^ Hostile(A)  =>  Criminal(Robert)

Query Criminal(Robert) proved: True

Proof tree:
Criminal(Robert)   <= R1 (crime law)
├── American(Robert)
├── Weapon(T1)   <= R3 (missile=>weapon)
│   └── Missile(T1)
├── Sells(Robert, T1, A)   <= R2 (missiles sold)
│   ├── Missile(T1)
│   └── Owns(A, T1)
└── Hostile(A)   <= R4 (enemy=>hostile)
    └── Enemy(A, America)
